# Grapheme-to-phoneme: Hungarian

The goal of this personal project
is to create a cascade of rules that maps from orthographical strings in Hungarian (```Input```) down to strings that represent their pronunciation (```Output```).

By convention, the input string appears on top and the output string on the bottom, as shown in the following example:

```
Input: cipőfűző
Output: tsipøːfyːzøː
```

The purpose of this task is to automatize the conversion from orthography to IPA (International Phonetic Alphabet).

## The Hungarian language

The Hungarian alphabet has 44 letters (14 vowels, 30 consonants).

```
Vowels: a	e	i	o	ö	u	ü á	é	í	ó	ő	ú	ű
Consonants: b c cs d dz dzs f g gy h j k l ly m n ny p q r s sz t ty v (w) (x) (y) z zs
```

#### **Vowel harmony**
The Hungarian language is characterized by vowel harmony, which means that the vowels in a word and in its suffixes need to harmonize. In practice, that means that the vowel in suffixes will change depending on the vowel in the root word. Vowels can be divided in two groups:
```
Mély magánhangzók (back vowels): a á o ó u ú
Magas magánhangzók (front vowels): e é i í ö ő ü	ű
```


# G2P

In [ ]:
!pip install pyfoma

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 6.8 MB/s  0:00:00
  Created wheel for pyfoma: filename=pyfoma-1.1.1-cp313-cp313-linux_x86_64.whl size=565180 sha256=3ad681ba8f0b8e59f239d551c3b52a4db04e3e6011c4250f581aabfd685e22af
  Stored in directory: /root/.cache/pip/wheels/04/2b/ba/7230650d169467e7a34793dbc3e88a35da1c2c5b195d2dee4f
Successfully built pyfoma
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyfoma]


In [ ]:
from pyfoma import *

fsts = {}
fsts['V'] = FST.re("a|e|i|o|ö|u|ü|á|é|í|ó|ő|ú|ű") # All the vowels
fsts['C'] = FST.re("b|c|(cs)|d|(dz)|(dzs)|f|g|(gy)|h|j|k|l|(ly)|m|n|(ny)|p|q|r|s|(sz)|t|(ty)|v|w|x|y|z|(zs)") # All the consonants

# Grammar rules
fsts['vowels'] = FST.re("$^rewrite(a:ɒ | e:ɛ | ö:ø | ü:y | á:(aː) | é:(eː) | í:(iː) | ó:(oː) | ő:(øː) | ú:(uː) | ű:(yː))")

fsts['crule'] = FST.re("$^rewrite(c:(ts))")
fsts['csrule'] = FST.re("$^rewrite((cs):(tʃ))")
fsts['dzsrule'] = FST.re("$^rewrite((dzs):(dʒ))")
fsts['gyrule'] = FST.re("$^rewrite((gy):ɟ)")
fsts['lyrule'] = FST.re("$^rewrite((ly):j)")
fsts['nyrule'] = FST.re("$^rewrite((ny):ɲ)")
fsts['srule'] = FST.re("$^rewrite(s:ʃ)")
fsts['szrule'] = FST.re("$^rewrite((sz):'&')")
fsts['szrule2'] = FST.re("$^rewrite('&':s)")
fsts['tyrule'] = FST.re("$^rewrite((ty):c)")
fsts['zsrule'] = FST.re("$^rewrite((zs):ʒ)")

# x qu w y
fsts['xrule'] = FST.re("$^rewrite(x:(ks))")
fsts['qrule'] = FST.re("$^rewrite((qu):k) | q:k")
fsts['wrule'] = FST.re("$^rewrite(w:v)")
fsts['yrule'] = FST.re("$^rewrite(y:i)")

# CC --> Cː
fsts['CC1'] = FST.re("$^rewrite((ccs):(Cː) | (ddz):(Dː) | (ddzs):(Jː) | (ggy):(Gː) | (lly):(Yː) | (nny):(Nː) | (ssz):(Sː) | (tty):(Tː) | (zzs):(Zː))")

fsts['CC2'] = FST.re(
    "$^rewrite("
    "(bb):(bː) | (cc):(cː) | (dd):(dː) | (ff):(fː) | (gg):(gː) | (hh):(hː) | "
    "(jj):(jː) | (kk):(kː) | (ll):(lː) | (mm):(mː) | (nn):(nː) | (pp):(pː) | "
    "(rr):(rː) | (ss):(sː) | (tt):(tː) | (vv):(vː) | (zz):(zː)"
    ")"
)


# Grammar
fsts['double'] = FST.re("$CC1 @ $CC2", fsts)

fsts['grammar'] = FST.re(
    "$szrule @ $zsrule @ $nyrule @ $lyrule @ "
    "$gyrule @ $dzsrule @ $csrule @ $zsrule @ $srule @ $crule @"
    "$szrule2 @ $qrule @ $xrule @ $wrule @ $yrule @ $vowels",
    fsts
)

fsts['lower'] = FST.re("$^rewrite((Cː):(cː) | (Dː):(dː) | (Jː):(jː) | (Gː):(gː) | (Yː):(yː) | "
"(Nː):(nː) | (Sː):(sː) |(Tː):(tː) | (Zː):(zː))")

fsts['clean'] = FST.re("$^rewrite((sʃ):(ʃː))")


# Full G2P
fsts['full'] = FST.re("$double @ $grammar @ $lower @ $clean", fsts)

# Test

In [ ]:
list(fsts['full'].generate("paci"))

['pɒtsi']

In [ ]:
# G2P large-scale automated testing
inputs = ["cipőfűző", "egészség", "megszentségteleníthetetlenségeskedéseitekkel"]
fsts['inputs'] = FST.re('|'.join([w for w in inputs]))
fsts['test'] = FST.re("$inputs @ $full", fsts)
print(Paradigm(fsts['test'], ".*"))

cipőfűző    tsipøːfyːzøː  
egészség    ɛgeːʃːeːg     

